# Diabetes Binary – Training and Evaluation

هذا الدفتر يدرّب نموذجاً تجريبياً على ملف المؤشرات الصحية الرقمي.

**تشغيله:** ضع هذا الملف وملف CSV في المجلد نفسه، ثم افتحه في VS Code/Jupyter وشغّل الخلايا بالترتيب.

- `Diabetes_binary` هو عمود الهدف (0 أو 1).
- نقسم البيانات إلى تدريب 60%، تحقق Validation 20%، واختبار Test 20%.
- نختار عتبة القرار من بيانات Validation فقط، ثم نقيّم مرة واحدة على Test.
- لا يمكن ضمان Recall أعلى من 90% للفئتين؛ النتائج الفعلية تعتمد على المعلومات الموجودة في الداتا.
- هذا نموذج تعليمي وليس أداة تشخيص طبي.

In [3]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, recall_score,
    balanced_accuracy_score, accuracy_score
)

In [4]:
data = pd.read_csv("diabetes_binary_health_indicators_BRFSS2015.csv")
print("Shape:", data.shape)
print("\nMissing values:", data.isnull().sum().sum())
print("\nTarget counts:")
print(data["Diabetes_binary"].value_counts())
print("\nColumns:")
print(data.columns.tolist())

Shape: (253680, 22)

Missing values: 0

Target counts:
Diabetes_binary
0.0    218334
1.0     35346
Name: count, dtype: int64

Columns:
['Diabetes_binary', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']


In [5]:
data = data.drop(columns=["Education","Income"])
X = data.drop(columns=["Diabetes_binary"])
y = data["Diabetes_binary"].astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (152208, 19)
Validation: (50736, 19)
Testing: (50736, 19)


In [6]:
model = HistGradientBoostingClassifier(
    max_iter=150,
    learning_rate=0.08,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    class_weight="balanced",
    random_state=42
)
model.fit(X_train, y_train)
print("Training completed.")

Training completed.


In [7]:
# نختار threshold من Validation فقط، بحيث نرفع أصغر Recall بين الفئتين.
val_probs = model.predict_proba(X_val)[:, 1]
thresholds = np.unique(np.quantile(val_probs, np.linspace(0, 1, 501)))

best = None
for threshold in thresholds:
    val_pred = (val_probs >= threshold).astype(int)
    recall_0 = recall_score(y_val, val_pred, pos_label=0)
    recall_1 = recall_score(y_val, val_pred, pos_label=1)
    candidate = (
        min(recall_0, recall_1),
        balanced_accuracy_score(y_val, val_pred),
        recall_0, recall_1, float(threshold)
    )
    if best is None or candidate[:2] > best[:2]:
        best = candidate

selected_threshold = best[4]
print(f"Selected threshold: {selected_threshold:.4f}")
print(f"Validation Recall class 0: {best[2]:.4f}")
print(f"Validation Recall class 1: {best[3]:.4f}")
print(f"Validation balanced accuracy: {best[1]:.4f}")

Selected threshold: 0.5420
Validation Recall class 0: 0.7471
Validation Recall class 1: 0.7489
Validation balanced accuracy: 0.7480


In [8]:
# الاختبار النهائي: لا نختار threshold من بيانات الاختبار.
test_probs = model.predict_proba(X_test)[:, 1]
y_pred = (test_probs >= selected_threshold).astype(int)

print("Classification report:")
print(classification_report(y_test, y_pred, digits=4))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print(f"Recall class 0: {recall_score(y_test, y_pred, pos_label=0):.4f}")
print(f"Recall class 1: {recall_score(y_test, y_pred, pos_label=1):.4f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_test, y_pred):.4f}")

Classification report:
              precision    recall  f1-score   support

           0     0.9488    0.7447    0.8345     43667
           1     0.3229    0.7519    0.4517      7069

    accuracy                         0.7457     50736
   macro avg     0.6358    0.7483    0.6431     50736
weighted avg     0.8616    0.7457    0.7812     50736

Confusion matrix:
[[32520 11147]
 [ 1754  5315]]
Recall class 0: 0.7447
Recall class 1: 0.7519
Accuracy: 0.7457
Balanced accuracy: 0.7483


In [9]:
# حفظ الموديل مع أسماء الأعمدة والـ threshold لكي يستخدمها الـ API لاحقاً.
bundle = {
    "model": model,
    "feature_columns": list(X.columns),
    "threshold": float(selected_threshold),
    "target_column": "Diabetes_binary",
    "model_name": "HistGradientBoostingClassifier",
    "notes": "Experimental educational model; not a medical diagnosis."
}
joblib.dump(bundle, "diabetes_model.joblib")
print("Saved: diabetes_model.joblib")

Saved: diabetes_model.joblib


In [10]:
# مثال بسيط لتحميل الموديل المحفوظ وعمل توقع لسجل واحد من الداتا.
loaded = joblib.load("diabetes_model.joblib")
sample = X_test.iloc[[0]][loaded["feature_columns"]]
probability_1 = loaded["model"].predict_proba(sample)[0, 1]
prediction = int(probability_1 >= loaded["threshold"])

print("Prediction:", prediction)
print(f"Estimated probability for class 1: {probability_1:.4f}")
print("Reminder: this is an experimental model output, not a medical diagnosis.")

Prediction: 1
Estimated probability for class 1: 0.8533
Reminder: this is an experimental model output, not a medical diagnosis.
